# 01 · Speech to text with Whisper

You will transcribe speech with **Whisper tiny**, a small encoder-decoder transformer: the encoder reads a log-mel spectrogram of the sound, the decoder writes the text token by token.

The model runs in this tab. **Download:** roughly 40 MB with 8-bit weights and roughly 150 MB with full-precision weights; which one is used depends on the device (the code below prints the real size). The browser caches it.

To use another model, change `MODEL` in the first cell.

In [ ]:
import time
import numpy as np
import lab

MODEL = "Xenova/whisper-tiny.en"     # English only. Try "Xenova/whisper-tiny" (multilingual) or "Xenova/whisper-base.en" (bigger)
AUDIO_URL = "https://huggingface.co/datasets/Xenova/transformers.js-docs/resolve/main/jfk.wav"
DEVICE = "auto"                      # "auto", "webgpu" (graphics card) or "wasm" (processor)

print("WebGPU available:", lab.device()["webgpu"])

## 1. Sound is a list of numbers

A microphone measures air pressure many times per second. `lab.load_audio` downloads a file, decodes it and returns the samples as a NumPy array with values between −1 and 1.

Whisper expects **16 000 samples per second** (16 kHz), so `load_audio` resamples to that rate by default.

In [ ]:
audio, sr = await lab.load_audio(AUDIO_URL)

print("samples:", audio.shape, audio.dtype)
print("sample rate:", sr, "Hz")
print("duration:", round(len(audio) / sr, 2), "seconds")
print("smallest and largest value:", round(float(audio.min()), 3), round(float(audio.max()), 3))
lab.play(audio, sr)

## 2. Load the model

`lab.pipeline` downloads the model the first time (watch the status line) and prepares it on the graphics card or the processor.

In [ ]:
asr = await lab.pipeline("automatic-speech-recognition", MODEL, device=DEVICE)

print(asr)
print("model files:", asr.size_mb, "MB, loaded in", asr.load_seconds, "s, running on", asr.device)

## 3. Transcribe

Calling the pipeline returns a dict with the key `"text"`. The **real-time factor** is processing time divided by the duration of the audio: below 1 means faster than real time. The very first call on a graphics card is slower than the following ones, because the browser compiles the model's programs.

In [ ]:
start = time.time()
result = await asr(audio)
seconds = time.time() - start

print(result["text"])
print(f"\n{seconds:.1f} s for {len(audio) / sr:.1f} s of audio: real-time factor {seconds / (len(audio) / sr):.2f}")

## 4. Timestamps

Whisper can also predict *when* something was said: at the beginning and the end of every stretch of speech, the decoder emits a special timestamp token. With `return_timestamps=True` the result has a list of `chunks`, each with a start time, an end time and its text.

In [ ]:
result = await asr(audio, return_timestamps=True)

lab.table([
    {"start (s)": chunk["timestamp"][0], "end (s)": chunk["timestamp"][1], "text": chunk["text"]}
    for chunk in result["chunks"]
])

## 5. Your turn: word error rate

How good is a transcript? The standard measure is the **word error rate**:

$$\mathrm{WER} = \frac{S + D + I}{N}$$

where $N$ is the number of words in the reference (the correct text) and $S$, $D$, $I$ are the numbers of substituted, deleted and inserted words in the best alignment of the transcript with the reference. $S + D + I$ is the **edit distance** (Levenshtein distance) between the two word lists. Because of insertions, the WER can be larger than 1.

The edit distance is computed with dynamic programming. Let $d[i][j]$ be the distance between the first $i$ reference words and the first $j$ hypothesis words. Then

- $d[i][0] = i$ (delete everything) and $d[0][j] = j$ (insert everything),
- $d[i][j] = \min\big(d[i-1][j] + 1,\; d[i][j-1] + 1,\; d[i-1][j-1] + c\big)$ with $c = 0$ if word $i$ of the reference equals word $j$ of the hypothesis, else $c = 1$.

Complete `word_error_rate` below. `normalize` is given: it lower-cases the text, removes punctuation and splits it into words.

In [ ]:
import re

def normalize(text):
    """Lower-case, remove punctuation, split into words."""
    return re.sub(r"[^a-z0-9' ]+", " ", text.lower()).split()

def word_error_rate(reference, hypothesis):
    """(substitutions + deletions + insertions) / number of words in the reference."""
    ref = normalize(reference)
    hyp = normalize(hypothesis)
    # TODO: fill the table d of shape (len(ref) + 1, len(hyp) + 1) as described above
    # and return d[len(ref)][len(hyp)] / len(ref).
    raise NotImplementedError("Complete word_error_rate, then run this cell and the check below.")

In [ ]:
assert word_error_rate("the cat sat on the mat", "the cat sat on the mat") == 0.0
assert word_error_rate("the cat sat on the mat", "The cat sat, on the mat!") == 0.0, "case and punctuation must not count"
assert word_error_rate("a b c d", "a x c d") == 0.25, "one substitution in four words"
assert word_error_rate("a b c d", "a c d") == 0.25, "one deletion in four words"
assert word_error_rate("a b c d", "a b x c d") == 0.25, "one insertion in four words"
assert word_error_rate("a b c d", "") == 1.0, "everything deleted"
assert word_error_rate("a b", "a x y b") == 1.0, "two insertions for two reference words"
assert word_error_rate("a b c", "c b a") == 2 / 3, "two substitutions are the best alignment here"
print("word_error_rate is correct.")

In [ ]:
REFERENCE = "And so my fellow Americans, ask not what your country can do for you, ask what you can do for your country."

print("reference: ", REFERENCE)
print("whisper:   ", result["text"].strip())
print("WER:       ", word_error_rate(REFERENCE, result["text"]))

## 6. Your own voice

`lab.record` records from your microphone. A dialog asks first, and the browser asks for permission the first time. Say one or two sentences, then compare what Whisper heard with what you said.

In [ ]:
my_audio, sr = await lab.record(5)      # seconds
lab.play(my_audio, sr)

mine = await asr(my_audio)
print(mine["text"])

In [ ]:
WHAT_I_SAID = "type here what you really said"

print("WER:", word_error_rate(WHAT_I_SAID, mine["text"]))

## Questions

1. Record the same sentence quietly, loudly, and with background noise. How does the WER change?
2. Whisper works on windows of 30 seconds. For longer recordings, pass `chunk_length_s=30` to the pipeline. What could go wrong at the border between two windows?
3. Change `MODEL` to a bigger Whisper and restart. Compare download size, real-time factor and WER.